# Python Pandas: แยกข้อมูล Excel เป็น Worksheet ตามค่าคอลัมน์ #5

**คำอธิบาย:** ใน Notebook นี้เราจะเรียนรู้วิธีแยกข้อมูล Excel เป็น Worksheet ตามค่าของคอลัมน์ คุณจะได้เรียนรู้วิธีแยกข้อมูลคอลัมน์ใน Excel เป็นหลาย Worksheet ใน Workbook เดียวกันด้วย Python Pandas

In [ ]:
# 📦 ติดตั้ง packages ที่จำเป็น
try:
    import pandas
    import openpyxl
    print("✅ ติดตั้ง packages ทั้งหมดแล้ว")
except ImportError:
    %pip install pandas openpyxl

Packages already installed.


## ขั้นตอนที่ 1: อ่านข้อมูล Excel
โหลดข้อมูลยอดขายจากไฟล์ Excel

In [ ]:
import requests
import os
import pandas as pd

url = "https://raw.githubusercontent.com/ddeshar/python-for-excel-manage/main/Random%20Sales%20Data.xlsx"
file_name = "Random Sales Data.xlsx"

# ตรวจสอบว่าไฟล์มีอยู่แล้วหรือไม่
if os.path.exists(file_name):
    print(f"✅ {file_name} มีอยู่แล้ว ข้ามการดาวน์โหลด")
else:
    print("📥 ไม่พบไฟล์ กำลังดาวน์โหลด...")

    r = requests.get(url)
    r.raise_for_status()

    with open(file_name, "wb") as f:
        f.write(r.content)

    print("✅ ดาวน์โหลดไฟล์สำเร็จ")

# อ่านไฟล์ Excel
df = pd.read_excel(file_name)

print(df.head())

File not found. Downloading...
File downloaded successfully.
  Sales Representative       Location Region         Customer  \
0          Sara Snyder  Massachusetts   East    Raymond Young   
1          Sara Snyder       New York   East       Helen Dean   
2       Diane Gonzalez     Washington   West   Shirley Chavez   
3          Sara Snyder     New Jersey   East       Brian Ryan   
4          Sara Snyder     New Jersey   East  Benjamin Willis   

            Order Date                                            Product  \
0  2016-01-01 00:00:00  Bravo II Megaboss 12-Amp Hard Body Upright, Re...   
1  2016-01-01 00:00:00  Hon Deluxe Fabric Upholstered Stacking Chairs,...   
2  2016-01-01 00:00:00  Acme Hot Forged Carbon Steel Scissors with Nic...   
3  2016-01-01 00:00:00      Bretford CR4500 Series Slim Rectangular Table   
4  2016-01-01 00:00:00                     Eldon Fold 'N Roll Cart System   

   Quantity  Price  Total Sale Amount  
0         6  12.42              74.52  
1    

In [ ]:
# ตรวจสอบคอลัมน์
print(df.columns.tolist())
print(f"ขนาดข้อมูล: {df.shape}")

['Sales Representative', 'Location', 'Region', 'Customer', 'Order Date', 'Product', 'Quantity', 'Price', 'Total Sale Amount']
Shape: (5684, 9)


## ขั้นตอนที่ 2: ดูค่าที่ไม่ซ้ำจากคอลัมน์
ระบุค่าที่ไม่ซ้ำในคอลัมน์ที่ต้องการแยก

In [ ]:
# ดูค่าที่ไม่ซ้ำจากคอลัมน์ที่จะแยก (เช่น 'Region' หรือ 'State')
# เปลี่ยน 'Region' เป็นชื่อคอลัมน์ที่ต้องการ
split_column = 'Region'
unique_values = df[split_column].unique()
print(f"ค่าที่ไม่ซ้ำใน '{split_column}': {unique_values}")
print(f"จำนวนค่าที่ไม่ซ้ำ: {len(unique_values)}")

Unique values in 'Region': ['East' 'West' nan]
Total unique values: 3


## ขั้นตอนที่ 3: แยกข้อมูลเป็นหลาย Worksheet
เขียนแต่ละกลุ่มข้อมูลไป Worksheet แยกใน Workbook เดียวกัน

In [ ]:
# แยกข้อมูลเป็นหลาย worksheet ตามค่าคอลัมน์
output_file = "Splited_Sales_Data.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for value in unique_values:
        # กรอง DataFrame สำหรับแต่ละค่าที่ไม่ซ้ำ
        filtered_df = df[df[split_column] == value]
        
        # เขียนไป worksheet ใหม่ที่ตั้งชื่อตามค่านั้น
        # ชื่อ Sheet ใน Excel ห้ามเกิน 31 ตัวอักษร
        sheet_name = str(value)[:31]
        filtered_df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"✍️ เขียน {len(filtered_df)} แถว ไป sheet: '{sheet_name}'")

print(f"\n✅ บันทึกไฟล์: {output_file}")

Written 3597 rows to sheet: 'East'
Written 2086 rows to sheet: 'West'
Written 0 rows to sheet: 'nan'

File saved: Splited_Sales_Data.xlsx


## ขั้นตอนที่ 4: ตรวจสอบผลลัพธ์
อ่านไฟล์ผลลัพธ์กลับมาเพื่อตรวจสอบว่าการแยกข้อมูลสำเร็จ

In [ ]:
# ตรวจสอบผลลัพธ์ - อ่านกลับมาตรวจสอบชื่อ sheet
xls = pd.ExcelFile(output_file)
print(f"Sheet ในไฟล์ผลลัพธ์: {xls.sheet_names}")

for sheet in xls.sheet_names:
    temp_df = pd.read_excel(output_file, sheet_name=sheet)
    print(f"  Sheet '{sheet}': {len(temp_df)} แถว")

Sheets in output file: ['East', 'West', 'nan']
  Sheet 'East': 3597 rows
  Sheet 'West': 2086 rows
  Sheet 'nan': 0 rows


---

## 🎮 Mini Project: แยกข้อมูลสินค้าออกเป็นไฟล์

ทดสอบการแยกข้อมูล Excel ด้วย pandas!

> 💡 **คำตอบ:** ดูไฟล์ `answers/07_answers.ipynb`

---

### โจทย์ที่ 1: แยกข้อมูลตาม Product Category 🟢
ใช้ไฟล์ `Random Sales Data.xlsx`:
1. อ่านข้อมูลด้วย `pd.read_excel()`
2. หา unique Product Category ทั้งหมด
3. แยกข้อมูลตาม Category ใส่แต่ละ Sheet ด้วย `ExcelWriter`
4. บันทึกเป็น `products_by_category.xlsx`
5. Print สรุป: Category + จำนวนแถว

> 💡 Hint: ใช้ `df[col].unique()` + `df[df[col]==value].to_excel(writer, sheet_name=...)`

In [ ]:
# โจทย์ที่ 1: แยกข้อมูลตาม Product Category
# เขียนโค้ดของคุณที่นี่ 👇



### โจทย์ที่ 2: แยกข้อมูลเป็นไฟล์แยก 🟡
1. อ่าน `Random Sales Data.xlsx`
2. แยกข้อมูลตาม `Region` — สร้าง **ไฟล์ Excel แยก** สำหรับแต่ละ Region
3. ตั้งชื่อไฟล์: `sales_East.xlsx`, `sales_West.xlsx`, ฯลฯ
4. ตรวจสอบว่าไฟล์ถูกสร้างครบ + print จำนวนแถวแต่ละไฟล์

> 💡 Hint: ใช้ loop + `filtered_df.to_excel(f"sales_{region}.xlsx", index=False)`

In [ ]:
# โจทย์ที่ 2: แยกข้อมูลเป็นไฟล์แยก
# เขียนโค้ดของคุณที่นี่ 👇

